# Week 4 — Day 4 | Manufacturing Plant Performance Analysis

**Python for Supply Chain Analytics — Week 4**

This notebook applies Pandas to a manufacturing supply chain scenario. The analysis focuses on practical business questions rather than Python syntax alone.


In [ ]:
import pandas as pd

## 1. Load the Dataset

Load the supplied CSV dataset into a Pandas DataFrame.

In [ ]:
production_df = pd.read_csv("Week_4_Day_4_Production.csv")
inventory_df = pd.read_csv("Week_4_Day_4_Inventory.csv")

## 2. Inspect the Data

Review the first records and confirm the dataset dimensions.

In [ ]:
production_df.head()

,Plant,Production_Line,Product,Planned_Production_Units,Actual_Production_Units,Machine_Available_Hours,Machine_Run_Hours,Production_Capacity_Units
0,Pune Manufacturing Unit,Line 1,Industrial Gear Housing,4200,4050,720,665,4300
1,Pune Manufacturing Unit,Line 2,Hydraulic Valve Body,3600,3420,680,610,3700
2,Pune Manufacturing Unit,Line 3,Electric Motor Casing,3900,3780,700,642,4050
3,Pune Manufacturing Unit,Line 4,Precision Coupling,3000,2910,620,570,3150
4,Vadodara Manufacturing Unit,Line 1,Industrial Gear Housing,4500,4410,740,690,4650


In [ ]:
inventory_df.head()

,Plant,Material,Closing_Inventory_Units,Safety_Stock_Units
0,Pune Manufacturing Unit,Cold Rolled Steel Sheet,235,260
1,Pune Manufacturing Unit,Polyamide Granules,265,300
2,Pune Manufacturing Unit,Copper Busbar,150,180
3,Pune Manufacturing Unit,Industrial Adhesive,120,160
4,Vadodara Manufacturing Unit,Aluminium Ingot,205,240


In [ ]:
print("Production Data:", production_df.shape)
print("Inventory Data:", inventory_df.shape)

Production Data: (16, 8)
Inventory Data: (16, 4)


## 3. Calculate Machine Utilization

Machine Utilization % = Machine Run Hours ÷ Available Hours × 100.

In [ ]:
production_df['Machine_Utilization_%'] = (production_df['Machine_Run_Hours'] / production_df['Machine_Available_Hours'] * 100)
production_df

,Plant,Production_Line,Product,Planned_Production_Units,Actual_Production_Units,Machine_Available_Hours,Machine_Run_Hours,Production_Capacity_Units,Machine_Utilization_%
0,Pune Manufacturing Unit,Line 1,Industrial Gear Housing,4200,4050,720,665,4300,92.361111
1,Pune Manufacturing Unit,Line 2,Hydraulic Valve Body,3600,3420,680,610,3700,89.705882
2,Pune Manufacturing Unit,Line 3,Electric Motor Casing,3900,3780,700,642,4050,91.714286
3,Pune Manufacturing Unit,Line 4,Precision Coupling,3000,2910,620,570,3150,91.935484
4,Vadodara Manufacturing Unit,Line 1,Industrial Gear Housing,4500,4410,740,690,4650,93.243243
5,Vadodara Manufacturing Unit,Line 2,Hydraulic Valve Body,3800,3650,690,618,3950,89.565217
6,Vadodara Manufacturing Unit,Line 3,Electric Motor Casing,4100,3895,710,650,4300,91.549296
7,Vadodara Manufacturing Unit,Line 4,Precision Coupling,3200,3040,640,585,3350,91.406250
8,Chennai Manufacturing Unit,Line 1,Industrial Gear Housing,4000,3760,700,625,4200,89.285714
9,Chennai Manufacturing Unit,Line 2,Hydraulic Valve Body,3500,3290,660,570,3650,86.363636


## 4. Calculate Inventory Gap

Inventory Gap = Safety Stock − Closing Inventory.

In [ ]:
inventory_df['Inventory_Gap_Units'] = (inventory_df['Safety_Stock_Units'] - inventory_df['Closing_Inventory_Units'])

In [ ]:
inventory_df

,Plant,Material,Closing_Inventory_Units,Safety_Stock_Units,Inventory_Gap_Units
0,Pune Manufacturing Unit,Cold Rolled Steel Sheet,235,260,25
1,Pune Manufacturing Unit,Polyamide Granules,265,300,35
2,Pune Manufacturing Unit,Copper Busbar,150,180,30
3,Pune Manufacturing Unit,Industrial Adhesive,120,160,40
4,Vadodara Manufacturing Unit,Aluminium Ingot,205,240,35
5,Vadodara Manufacturing Unit,Engineering Polymer Pellets,315,350,35
6,Vadodara Manufacturing Unit,Silicone Sealant,135,160,25
7,Vadodara Manufacturing Unit,Stainless Steel Coil,220,250,30
8,Chennai Manufacturing Unit,Electrical Grade Steel,225,300,75
9,Chennai Manufacturing Unit,Glass Fibre Reinforced Nylon,250,280,30


## 5. Calculate Machine Utilization

Machine Utilization % = Machine Run Hours ÷ Available Hours × 100.

In [ ]:
plant_utilization = (production_df.groupby('Plant')['Machine_Utilization_%'].mean())
plant_utilization

,Machine_Utilization_%
Plant,
Chennai Manufacturing Unit,88.988318
Nashik Manufacturing Unit,91.279003
Pune Manufacturing Unit,91.429191
Vadodara Manufacturing Unit,91.441002


## 6. Aggregate Inventory Gap by Plant

In [ ]:
plant_inventory_gap = (inventory_df.groupby('Plant')['Inventory_Gap_Units'].sum())
plant_inventory_gap

,Inventory_Gap_Units
Plant,
Chennai Manufacturing Unit,180
Nashik Manufacturing Unit,5
Pune Manufacturing Unit,130
Vadodara Manufacturing Unit,125


## 7. Average Machine Utilization by Plant

In [ ]:
plant_risk_summary = pd.merge(plant_utilization,plant_inventory_gap,left_index=True,right_index=True)
plant_risk_summary

,Machine_Utilization_%,Inventory_Gap_Units
Plant,,
Chennai Manufacturing Unit,88.988318,180
Nashik Manufacturing Unit,91.279003,5
Pune Manufacturing Unit,91.429191,130
Vadodara Manufacturing Unit,91.441002,125


## 8. Build Plant Production Performance

Aggregate planned and actual production and calculate achievement.

In [ ]:
plant_production_summary = production_df.groupby('Plant')[['Planned_Production_Units', 'Actual_Production_Units']].sum()
plant_production_summary['Production_Achievement_%'] = (plant_production_summary['Actual_Production_Units']/plant_production_summary['Planned_Production_Units'] * 100)
plant_production_summary

,Planned_Production_Units,Actual_Production_Units,Production_Achievement_%
Plant,,,
Chennai Manufacturing Unit,14100,13290,94.255319
Nashik Manufacturing Unit,15100,14505,96.059603
Pune Manufacturing Unit,14700,14160,96.326531
Vadodara Manufacturing Unit,15600,14995,96.121795


## 9. Combine Plant-Level Utilization and Inventory

Use `merge()` to combine independently aggregated plant metrics.

In [ ]:
plant_performance = pd.merge(plant_risk_summary, plant_production_summary, left_index=True, right_index=True)
plant_performance

,Machine_Utilization_%,Inventory_Gap_Units,Planned_Production_Units,Actual_Production_Units,Production_Achievement_%
Plant,,,,,
Chennai Manufacturing Unit,88.988318,180,14100,13290,94.255319
Nashik Manufacturing Unit,91.279003,5,15100,14505,96.059603
Pune Manufacturing Unit,91.429191,130,14700,14160,96.326531
Vadodara Manufacturing Unit,91.441002,125,15600,14995,96.121795


## 10. Consolidated Plant Performance

Combine production, utilization, and inventory indicators into one plant-level view.

In [ ]:
attention_plants = plant_performance[plant_performance['Production_Achievement_%'] < 95]
attention_plants

,Machine_Utilization_%,Inventory_Gap_Units,Planned_Production_Units,Actual_Production_Units,Production_Achievement_%
Plant,,,,,
Chennai Manufacturing Unit,88.988318,180,14100,13290,94.255319


## 11. Calculate Machine Utilization

Machine Utilization % = Machine Run Hours ÷ Available Hours × 100.

In [ ]:
attention_summary = attention_plants[['Production_Achievement_%','Machine_Utilization_%','Inventory_Gap_Units']]
attention_summary

,Production_Achievement_%,Machine_Utilization_%,Inventory_Gap_Units
Plant,,,
Chennai Manufacturing Unit,94.255319,88.988318,180


## Business Takeaway

The analysis integrates production, machine utilization, and inventory indicators at plant level. It demonstrates how Pandas can combine multiple operational datasets to create a focused manufacturing performance view.